In [104]:
import torch
from pathlib import Path

import numpy as np
from omegaconf import OmegaConf
from torch.utils.data import DataLoader
from typing import Dict, Tuple

from uncertainty_estimation.geometry.stereo import reproject
from uncertainty_estimation.training.data.semistaticsim import (
    SemiStaticSimStereoDataset,
    stereo_collate,
)
from uncertainty_estimation.training.trainer import _get_depth, _in_bounds

from uncertainty_estimation.matching.orb import ORB as ORBMatcher
from uncertainty_estimation.matching.orb import SIFT 
import matplotlib.pyplot as plt


def unproject(kps_px, depth, K):
    """kps_px: (B, N, 2), depth: (B, N), K: (B, 3, 3) -> (B, N, 3) in camera frame."""
    B, N, _ = kps_px.shape
    ones = torch.ones(B, N, 1, device=kps_px.device, dtype=kps_px.dtype)
    kps_h = torch.cat([kps_px, ones], dim=-1)                      # (B, N, 3)
    K_inv = torch.linalg.inv(K)                                     # (B, 3, 3)
    rays  = torch.einsum("bij,bnj->bni", K_inv, kps_h)              # (B, N, 3), z=1
    return rays * depth.unsqueeze(-1)                               # scale by Z

def transform_points(X, T):
    """X: (B, N, 3), T: (B, 4, 4) -> T @ X in homogeneous."""
    B, N, _ = X.shape
    ones = torch.ones(B, N, 1, device=X.device, dtype=X.dtype)
    Xh = torch.cat([X, ones], dim=-1)                               # (B, N, 4)
    return torch.einsum("bij,bnj->bni", T, Xh)[..., :3]

In [107]:
from torch import Tensor

def _lookup_depth(depth_map: Tensor, kps: Tensor) -> Tensor:
    """Sample a dense depth map at keypoint locations (nearest-neighbour).

    Args:
        depth_map: (B, H, W) depth in metres.
        kps:       (B, P, 2) keypoints in pixel coords (x=col, y=row).

    Returns:
        (B, P) depth values at each keypoint.
    """
    B, H, W = depth_map.shape
    col = kps[..., 0].round().long().clamp(0, W - 1) # (B, P)
    row = kps[..., 1].round().long().clamp(0, H - 1) # (B, P)
    b_idx = torch.arange(B, device=kps.device).unsqueeze(1).expand_as(col) # (B, P)
    return depth_map[b_idx, row, col] # now we can have depth_map[b_idx[i, j], row[i, j], col[i, j]] for each keypoint

def simulate(
    images, depth_maps, K, T_lr,
    sigma_det_px: float,
    sigma_Z_m: float,
    n_kps: int = 500,
):
    device = depth_maps.device
    dtype = depth_maps.dtype
    B = images.shape[0]
    H, W = images.shape[-2:]
    T_rl = torch.linalg.inv(T_lr)

    # 1. Sample uniform random left-image locations
    u_L = torch.rand(B, n_kps, 2, device=device, dtype=dtype) * \
          torch.tensor([W - 1, H - 1], device=device, dtype=dtype)

    # 2. Look up GT depth at left keypoints
    Z_L = _lookup_depth(depth_maps[:, 0], u_L)
    valid = (Z_L > 0.5) & (Z_L < 50.0)

    # 3. GT right correspondence by projection
    u_R_gt = reproject(u_L, Z_L, K, T_lr)

    in_bounds_R = (u_R_gt[..., 0] >= 0) & (u_R_gt[..., 0] < W) & \
                  (u_R_gt[..., 1] >= 0) & (u_R_gt[..., 1] < H)
    valid = valid & in_bounds_R

    Z_R_gt = _lookup_depth(depth_maps[:, 1], u_R_gt)
    valid = valid & (Z_R_gt > 0.5) & (Z_R_gt < 50.0)

    # 4. Inject independent noise
    eps1 = sigma_det_px * torch.randn_like(u_L)
    eps2 = sigma_det_px * torch.randn_like(u_R_gt)
    dZ_L = sigma_Z_m * torch.randn_like(Z_L)
    dZ_R = sigma_Z_m * torch.randn_like(Z_R_gt)

    u_L_noisy = u_L + eps1
    u_R_noisy = u_R_gt + eps2
    Z_L_noisy = Z_L + dZ_L
    Z_R_noisy = Z_R_gt + dZ_R

    # 5. Pixel residual as seen by the loss
    u_R_reproj = reproject(u_L_noisy, Z_L_noisy, K, T_lr)
    r_2d = u_R_noisy - u_R_reproj

    # 6. 3D residual in left frame
    X_L_recon = unproject(u_L_noisy, Z_L_noisy, K)
    X_R_recon = unproject(u_R_noisy, Z_R_noisy, K)
    X_R_in_L = transform_points(X_R_recon, T_rl)
    r_3d = X_L_recon - X_R_in_L

    return r_2d[valid], r_3d[valid], Z_L[valid], u_L[valid]

In [109]:
NOISE_CONDITIONS = {
    "detector_only":   dict(sigma_det_px=0.5, sigma_Z_m=0.0),
    "depth_only":      dict(sigma_det_px=0.0, sigma_Z_m=0.1),
    "both":            dict(sigma_det_px=0.5, sigma_Z_m=0.1),
    "zero_sanity":     dict(sigma_det_px=0.0, sigma_Z_m=0.0),  # run this first
}

N_KPS       = 500
MAX_SAMPLES = 500
BATCH_SIZE  = 8
ORIENTATIONS = ["horizontal", "vertical"]
BASELINES    = [5, 10, 20, 50, 100]
DEVICE           = "cpu"

repo_root   = "/Users/adam/Documents/MILA/projects/uncertainty_estimation"

# Nested dict: condition -> stereo -> (r_2d, r_3d, Z_L, u_L) as numpy arrays
all_results = {cond: {} for cond in NOISE_CONDITIONS}

for orient in ORIENTATIONS:
    for b in BASELINES:
        stereo = f"{orient}_{b}cm"
        base    = OmegaConf.load(f"{repo_root}/configs/base.yaml")
        dataset = OmegaConf.load(f"{repo_root}/configs/dataset/sss.yaml")
        cfg     = OmegaConf.merge(base, {"dataset": dataset})
        cfg.dataset.stereo_config = stereo

        # matching_cfg=None so __getitem__ doesn't run ORB — we don't need matches
        ds = SemiStaticSimStereoDataset(cfg.dataset, cfg.augmentation, "train", None)
        n_take = min(MAX_SAMPLES, len(ds))
        subset = torch.utils.data.Subset(ds, list(range(n_take)))
        loader = DataLoader(subset, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=0, collate_fn=stereo_collate)

        # Accumulate per-condition chunks across batches
        chunks = {cond: {"r_2d": [], "r_3d": [], "Z_L": [], "u_L": []} 
                  for cond in NOISE_CONDITIONS}

        with torch.no_grad():
            for batch in loader:
                images     = batch["images"].to(DEVICE)       # (B, 2, C, H, W)
                K          = torch.linalg.inv(batch["K_inv"]).to(DEVICE)
                T_lr       = batch["T_lr"].to(DEVICE)
                depth_maps = torch.stack([
                    batch["depth_left"].to(DEVICE),
                    batch["depth_right"].to(DEVICE),
                ], dim=1)                                      # (B, 2, H, W)

                for cond_name, noise_kwargs in NOISE_CONDITIONS.items():
                    r_2d, r_3d, Z_L, u_L = simulate(
                        images, depth_maps, K, T_lr,
                        n_kps=N_KPS, **noise_kwargs,
                    )
                    chunks[cond_name]["r_2d"].append(r_2d.cpu().numpy())
                    chunks[cond_name]["r_3d"].append(r_3d.cpu().numpy())
                    chunks[cond_name]["Z_L"].append(Z_L.cpu().numpy())
                    chunks[cond_name]["u_L"].append(u_L.cpu().numpy())

        for cond_name in NOISE_CONDITIONS:
            all_results[cond_name][stereo] = {
                k: np.concatenate(v) for k, v in chunks[cond_name].items()
            }
            r_2d = all_results[cond_name][stereo]["r_2d"]
            print(f"{cond_name:15s}  {stereo:20s}  N={len(r_2d):6d}  "
                  f"σx={r_2d[:,0].std():.3f}  σy={r_2d[:,1].std():.3f}")

detector_only    horizontal_5cm        N=237375  σx=0.709  σy=0.706
depth_only       horizontal_5cm        N=237464  σx=1.571  σy=0.000
both             horizontal_5cm        N=237531  σx=1.728  σy=0.709
zero_sanity      horizontal_5cm        N=237387  σx=0.000  σy=0.000
detector_only    horizontal_10cm       N=232796  σx=0.708  σy=0.709
depth_only       horizontal_10cm       N=232725  σx=3.044  σy=0.000
both             horizontal_10cm       N=232647  σx=3.133  σy=0.708
zero_sanity      horizontal_10cm       N=232893  σx=0.000  σy=0.000
detector_only    horizontal_20cm       N=223352  σx=0.706  σy=0.707
depth_only       horizontal_20cm       N=222834  σx=5.694  σy=0.000
both             horizontal_20cm       N=223013  σx=5.684  σy=0.709
zero_sanity      horizontal_20cm       N=222897  σx=0.000  σy=0.000
detector_only    horizontal_50cm       N=192916  σx=0.707  σy=0.708
depth_only       horizontal_50cm       N=192722  σx=12.269  σy=0.000
both             horizontal_50cm       N=192425